# 09 — Full Training Pipeline


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

print("Project root:", ROOT)


In [ ]:
import os

SKIP_TRANSFORMERS = os.environ.get("SKIP_TRANSFORMERS", "0") == "1"
cmd = [sys.executable, str(ROOT / "scripts" / "run_full_training.py")]
if SKIP_TRANSFORMERS:
    cmd.append("--skip-transformers")

print("Command:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT), capture_output=False, text=True)
print(f"Exit code: {result.returncode}")


## Unified model comparison


In [ ]:
from IPython.display import display
results_dir = ROOT / "outputs" / "results"

if (results_dir / "all_models_comparison.csv").exists():
    df = pd.read_csv(results_dir / "all_models_comparison.csv")
    print(f"Total models: {len(df)}")
    if "tier" in df.columns:
        display(df.groupby("tier")["accuracy"].agg(["count", "mean", "max"]).round(3))
    display(df.sort_values("accuracy", ascending=False).head(20))
else:
    print("Run training first — all_models_comparison.csv not found.")


In [ ]:
from IPython.display import display
for chart in ["all_models_comparison.png", "fusion_strategy_comparison.png", "paradox_scatter.png"]:
    path = results_dir / chart
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Missing: {chart}")


## Per-modality results


In [ ]:
from IPython.display import display
for csv_name in [
    "model_comparison.csv",
    "text_model_results.csv",
    "image_model_results.csv",
    "trimodal_results.csv",
    "fusion_results.csv",
    "transformer_results.csv",
]:
    path = results_dir / csv_name
    if path.exists():
        print(f"\n=== {csv_name} ===")
        display(pd.read_csv(path))


In [ ]:
from IPython.display import display
report_path = results_dir / "full_training_report.json"
if report_path.exists():
    with report_path.open(encoding="utf-8") as handle:
        report = json.load(handle)
    print(f"Generated: {report.get('generated_at', 'N/A')}")
    print(f"Total models: {report.get('total_models', 'N/A')}")
    print(f"Mean accuracy: {report.get('mean_accuracy', 'N/A')}")
    if report.get("top_15"):
        print("\nTop 15 models:")
        display(pd.DataFrame(report["top_15"]))
else:
    print("full_training_report.json not found yet.")

print("09_full_training.py complete.")
